In [9]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

In [2]:
fold_1_train = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_1_train.csv')
fold_2_train = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_2_train.csv')
fold_3_train = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_3_train.csv')
fold_4_train = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_4_train.csv')
fold_5_train = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_5_train.csv')

fold_1_val = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_1_val.csv')
fold_2_val = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_2_val.csv')
fold_3_val = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_3_val.csv')
fold_4_val = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_4_val.csv')
fold_5_val = pd.read_csv('/Users/joanwong/Desktop/dsa4263/DSA4263 Project/data/preprocessed/fold_5_val.csv')

In [3]:
train_folds = [fold_1_train, fold_2_train, fold_3_train, fold_4_train, fold_5_train]
val_folds = [fold_1_val, fold_2_val, fold_3_val, fold_4_val, fold_5_val]

In [4]:
# Concatenate SMOTEd folds 1-4 for training
training_data = pd.concat([fold_1_train, fold_2_train, fold_3_train, fold_4_train])
X_train = training_data.drop(columns='fraud_bool')
y_train = training_data['fraud_bool']

# Use fold 5 as validation
X_val = fold_5_val.drop(columns='fraud_bool') 
y_val = fold_5_val['fraud_bool']

In [5]:
# Step 2: Initialize the Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# Step 3: Train the model on the training data
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, random_state=42)

In [6]:
# Step 4: Evaluate the model on the validation set
y_pred = rf.predict(X_val)

# Classification report for precision, recall, F1-score, and accuracy
print(classification_report(y_val, y_pred))

# Confusion matrix for false positives and false negatives
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97    178711
           1       0.08      0.41      0.14      1920

    accuracy                           0.95    180631
   macro avg       0.54      0.68      0.56    180631
weighted avg       0.98      0.95      0.96    180631

[[170145   8566]
 [  1133    787]]


In [7]:
# ROC AUC score for model performance
roc_auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
print(f'ROC AUC: {roc_auc:.2f}')

ROC AUC: 0.85


In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix, f1_score
import pandas as pd
import os
import matplotlib.pyplot as plt

# Define the objective function for Optuna
def objective(trial):
    # Hyperparameters to tune
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),  # Number of trees
        'max_depth': trial.suggest_int('max_depth', 10, 30),  # Maximum depth of the tree
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),  # Min samples required to split a node
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),  # Min samples required to be at a leaf node
        'max_features': trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2'])  # Number of features to consider
    }

    # Create the Random Forest model with the suggested hyperparameters
    rf = RandomForestClassifier(**param, random_state=42)

    # Initialize metrics tracking
    auc_roc = 0
    auc_pr = 0
    f1_score_avg = 0
    fold_classification_report = ""

    # Loop through the 5 folds
    for i in range(5):
        # Load training and validation data for the current fold
        fold_train_data = pd.read_csv(f"/path/to/fold_{i+1}_train.csv")
        fold_val_data = pd.read_csv(f"/path/to/fold_{i+1}_val.csv")

        # Separate features and target variable
        X_train_fold = fold_train_data.drop(columns='fraud_bool')
        y_train_fold = fold_train_data['fraud_bool']
        X_val_fold = fold_val_data.drop(columns='fraud_bool')
        y_val_fold = fold_val_data['fraud_bool']

        # Fit the model with the current fold's data
        rf.fit(X_train_fold, y_train_fold)

        # Predict and calculate the probabilities on the validation set
        y_pred_fold = rf.predict(X_val_fold)
        y_proba_fold = rf.predict_proba(X_val_fold)[:, 1]  # Get probability for the positive class (fraud)

        # Calculate the evaluation metrics
        auc_roc += roc_auc_score(y_val_fold, y_proba_fold)
        auc_pr += average_precision_score(y_val_fold, y_proba_fold)
        f1_score_avg += f1_score(y_val_fold, y_pred_fold)

        # Get classification report for the fold
        fold_classification_report += f"\nFold {i+1} Classification Report:\n"
        fold_classification_report += classification_report(y_val_fold, y_pred_fold)

    # Calculate the average AUC-ROC, AUC-PR, and F1-score across all folds
    auc_roc /= 5
    auc_pr /= 5
    f1_score_avg /= 5

    # Print metrics
    print(f"Average AUC-ROC: {auc_roc:.4f}")
    print(f"Average AUC-PR: {auc_pr:.4f}")
    print(f"Average F1-Score: {f1_score_avg:.4f}")
    print(fold_classification_report)

    # Return the AUC-PR score as the objective to maximize
    return auc_roc  

In [ ]:
# Create an Optuna study and optimize the objective
study = optuna.create_study(direction='maximize')  # Maximize AUC-ROC
study.optimize(objective, n_trials=50)  # Perform 50 trials of optimization

# Print the best parameters and the corresponding AUC-ROC score
print(f"Best hyperparameters: {study.best_params}")
print(f"Best AUC-ROC: {study.best_value:.4f}")

In [ ]:
# Retraining the final model with the best hyperparameters on the entire training dataset
# Load the entire training data (from all 5 folds) to retrain the model
final_train_data = pd.concat([pd.read_csv(f"/path/to/fold_{i+1}_train.csv") for i in range(5)])
X_train_final = final_train_data.drop(columns='fraud_bool')
y_train_final = final_train_data['fraud_bool']

# Initialize Random Forest with the best hyperparameters
final_rf = RandomForestClassifier(**study.best_params, random_state=42)

# Fit the model with the entire training data
final_rf.fit(X_train_final, y_train_final)

In [ ]:
# Load the test set (assuming you have a separate test set)
test_data = pd.read_csv("/path/to/test_data.csv")
X_test = test_data.drop(columns='fraud_bool')
y_test = test_data['fraud_bool']

In [ ]:
# Make predictions on the test set
y_pred_test = final_rf.predict(X_test)
y_proba_test = final_rf.predict_proba(X_test)[:, 1]  # Get probability for the positive class (fraud)

In [ ]:
# Print classification report and confusion matrix for the test set
print("\nFinal Model Evaluation on Test Set:")
print(classification_report(y_test, y_pred_test))
print(f"ROC AUC on Test Set: {roc_auc_score(y_test, y_proba_test):.4f}")

# Print confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
print("Confusion Matrix:")
print(cm)

# Optional: Plot Confusion Matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = range(2)
plt.xticks(tick_marks, ['Non-Fraud', 'Fraud'], rotation=45)
plt.yticks(tick_marks, ['Non-Fraud', 'Fraud'])
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()
